# Khmer History RAG — User-Side Chat Pipeline

This notebook is the **user-facing** half of the project. The chunks are already
built (`rag_store.json`) and the retriever embedding model is already fine-tuned
(`khmer-retriever-finetuned/`). Here we turn that into an interactive chat.

## Pipeline
```
User asks in Khmer (Gradio textbox)
   -> embed_query()   : model = khmer-retriever-finetuned (e5 fine-tuned, 384-dim)
                        text  = "query: " + question
   -> dense_search()  : cosine(query_vec, chunk_matrix)  -> semantic scores
   -> bm25_search()   : khmer-nltk tokenize -> keyword scores
   -> retrieve()      : HYBRID = alpha*dense + (1-alpha)*bm25, sort, take top_k
   -> generate()      : Gemini writes a grounded Khmer answer from the top chunks
   -> Gradio chat     : shows answer + cited sources, keeps history
```

**Why hybrid?** The evaluation (`eval_results_comparison.csv`) showed Hybrid
(BM25 + fine-tuned e5) had the best **paraphrased** R@1 (0.424) and MRR (0.542).
Real users type in their own words (paraphrased), so hybrid is the right choice.

**Chunks are re-encoded at startup** with the loaded model so queries and chunks
share the exact same encode path (the cached `emb_ft.npy` was produced by a
slightly different model/library version and is not bit-identical).

## 1. Imports & configuration

In [1]:
import os
import json
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from khmernltk import word_tokenize
import gradio as gr

# ---- paths ----
ROOT        = Path.cwd()
STORE_PATH  = ROOT / "rag_store.json"
MODEL_PATH  = ROOT / "khmer-retriever-finetuned"

# ---- retrieval knobs ----
TOP_K = 5        # how many chunks to feed the LLM
ALPHA = 0.5      # weight on dense (semantic); (1-ALPHA) on BM25 (lexical)

# ---- e5 prefixes (this model was fine-tuned with these; see its README) ----
QUERY_PREFIX   = "query: "
PASSAGE_PREFIX = "passage: "

# ---- Gemini ----
GEMINI_MODEL = "gemini-2.5-flash"   # fast, free-tier friendly, good Khmer
print("Config loaded. ROOT =", ROOT)

d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config loaded. ROOT = d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation


## 2. Load the Gemini API key (securely)

The key is read from the `GEMINI_API_KEY` environment variable, loaded from a
git-ignored `.env` file. **Never hardcode the key in this notebook.**

Create a file named `.env` in the project root containing one line:
```
GEMINI_API_KEY=your_real_key_here
```
(See `.env.example`.) If you skip this, retrieval still works but generation is
disabled and the app falls back to showing the raw retrieved chunks.

In [2]:
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")   # loads GEMINI_API_KEY into the environment

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

gemini_client = None
if GEMINI_API_KEY:
    from google import genai
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client ready (key loaded from environment).")
else:
    print("WARNING: GEMINI_API_KEY not found. Generation disabled -> will show raw chunks.")
    print("Create a .env file with GEMINI_API_KEY=... to enable Gemini answers.")

Gemini client ready (key loaded from environment).


## 3. Load the store and the retriever model

In [3]:
with open(STORE_PATH, encoding="utf-8") as f:
    store = json.load(f)

chunks   = store["chunks"]                              # list of 908 chunk dicts
texts    = [c["text_payload"] for c in chunks]          # raw Khmer text
metas    = [c["metadata"] for c in chunks]              # page_title, source_url, ...
chunk_ids = [c["chunk_id"] for c in chunks]

print(f"Loaded {len(chunks)} chunks.")

model = SentenceTransformer(str(MODEL_PATH))
print("Retriever model loaded:", MODEL_PATH.name)

Loaded 908 chunks.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4261.09it/s]


Retriever model loaded: khmer-retriever-finetuned


## 4. Re-encode all chunks (dense index)

We encode every chunk now, with the loaded model, so the chunk vectors and the
query vectors come from the identical encode path. `normalize_embeddings=True`
means a dot product equals cosine similarity. Runs in a few seconds on GPU.

In [4]:
chunk_emb = model.encode(
    [PASSAGE_PREFIX + t for t in texts],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)
chunk_emb = np.asarray(chunk_emb, dtype=np.float32)
print("Dense index shape:", chunk_emb.shape)   # (908, 384)

Batches: 100%|██████████| 29/29 [00:04<00:00,  7.05it/s]

Dense index shape: (908, 384)


## 5. Build the BM25 (lexical) index

BM25 uses **no embeddings** — it matches overlapping words. We tokenize each
chunk with the Khmer tokenizer so word boundaries are correct (Khmer has no
spaces between words).

In [5]:
tokenized_chunks = [word_tokenize(t) for t in texts]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 index built over", len(tokenized_chunks), "chunks.")

| 2026-11-03 09:46:30,319 | INFO | khmer-nltk | Loaded model from d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\khmernltk\word_tokenize\sklearn_crf_ner_10000.sav |


BM25 index built over 908 chunks.


## 6. Retrieval functions (embed query -> search -> rank)

In [6]:
def embed_query(question: str) -> np.ndarray:
    """Embed the user question with the SAME model as the chunks."""
    return model.encode(
        QUERY_PREFIX + question,
        normalize_embeddings=True,
    ).astype(np.float32)


def dense_search(question: str) -> np.ndarray:
    """Semantic / embedding search: cosine similarity vs every chunk."""
    q = embed_query(question)
    return chunk_emb @ q               # (908,) similarity scores


def bm25_search(question: str) -> np.ndarray:
    """Lexical search: keyword-overlap scores."""
    return np.asarray(bm25.get_scores(word_tokenize(question)), dtype=np.float32)


def _minmax(x: np.ndarray) -> np.ndarray:
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


def retrieve(question: str, top_k: int = TOP_K, alpha: float = ALPHA):
    """HYBRID retrieval: combine dense + BM25, rank, return top_k chunks."""
    d = _minmax(dense_search(question))
    b = _minmax(bm25_search(question))
    combined = alpha * d + (1 - alpha) * b
    top_idx = np.argsort(combined)[::-1][:top_k]
    return [
        {
            "chunk_id": chunk_ids[i],
            "text":     texts[i],
            "page":     metas[i].get("page_title", ""),
            "url":      metas[i].get("source_url", ""),
            "score":    float(combined[i]),
        }
        for i in top_idx
    ]

## 7. Generation (Gemini writes the grounded Khmer answer)

We stuff the retrieved chunks into a prompt and instruct Gemini to answer **only**
from that context, in Khmer. If no API key is set, we fall back to returning the
top chunks directly so the pipeline still demonstrates retrieval.

In [10]:
SYSTEM_INSTRUCTION = (
    "You are a helpful assistant that answers questions about Khmer (Cambodian) "
    "history. Answer ONLY using the provided context passages. If the answer is "
    "not in the context, say in Khmer that you do not have enough information "
    "(\u179f\u17bc\u1798\u1791\u17c4\u179f \u1781\u17d2\u1789\u17bb\u17c6\u1798\u17b7\u1793\u1798\u17b6\u1793\u1796\u17d0\u178f\u17cc\u1798\u17b6\u1793\u1782\u17d2\u179a\u1794\u17cb\u1782\u17d2\u179a\u17b6\u1793\u17cb). "
    "Always answer in Khmer language. The max words no more than 120 words"
)


def build_context(hits):
    """Format retrieved chunks into a numbered context block."""
    blocks = []
    for n, h in enumerate(hits, 1):
        blocks.append(f"[\u17a1{n}] ({h['page']})\n{h['text']}")
    return "\n\n".join(blocks)


def generate(question: str, hits) -> str:
    """Produce a grounded Khmer answer. Falls back to raw chunks if no LLM."""
    context = build_context(hits)

    if gemini_client is None:
        # No API key -> show retrieval result directly.
        return "⚠️ (Gemini off — រកឃើញឯកសារពាក់ព័ន្ធ):\n\n" + context

    prompt = (
        f"កម្មវត្ថុ (Context):\n{context}\n\n"
        f"សំណួរ (Question):\n{question}\n\n"
        f"សូមឆ្លើយជាភាសាខ្មែរ ដោយផ្អែកលើព័ត៌មានខាងលើប៉ុណ្ណោះ។"
    )

    resp = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={"system_instruction": SYSTEM_INSTRUCTION, "temperature": 0.2},
    )
    # return only the grounded answer (no sources block)
    return resp.text.strip()


# minimum top-score for a query to be treated as a real history question.
# below this the retrieval found nothing relevant (e.g. greetings / noise).
RELEVANCE_THRESHOLD = 0.60
NO_INFO_MSG = "សូមទោស ខ្ញុំមិនមានព័ត៌មានគ្រប់គ្រាន់ដើម្បីឆ្លើយសំណួរនេះទេ។ សូមសួរអំពីប្រវត្តិសាស្ត្រកម្ពុជា។"


def answer_question(question: str) -> str:
    """Full RAG: retrieve -> relevance gate -> generate."""
    question = (question or "").strip()
    if not question:
        return NO_INFO_MSG

    hits = retrieve(question)
    top_score = hits[0]["score"] if hits else 0.0

    # If nothing is relevant enough (e.g. 'hi', 'jom'), don't call the LLM
    # and don't show sources — just a friendly redirect.
    if top_score < RELEVANCE_THRESHOLD:
        return NO_INFO_MSG

    return generate(question, hits)

## 8. Quick test (before launching the UI)

In [11]:
q = "តើអង្គរវត្តត្រូវបានសាងសង់នៅសតវត្សទីប៉ុន្មាន?"
print("Q:", q, "\n")
print(answer_question(q))

Q: តើអង្គរវត្តត្រូវបានសាងសង់នៅសតវត្សទីប៉ុន្មាន? 

ប្រាសាទអង្គរវត្តត្រូវបានសាងសង់ឡើងក្នុងសតវត្សទី១២។


## 9. Gradio chat interface

`fn` receives the latest message and the chat history. We ignore history for
retrieval (each question is retrieved independently) but Gradio still shows the
running conversation.

In [12]:

def chat_fn(message, history):
    return answer_question(message)


# ---- load the Kh-Menghorn Khmer font and embed it as CSS ----
import base64

FONT_PATH = ROOT / "fonts" / "Kh-Menghorn.ttf"   # put the .ttf here
FONT_FAMILY = "Kh-Menghorn"

def build_font_css():
    if not FONT_PATH.exists():
        print(f"NOTE: font not found at {FONT_PATH} -> using default Khmer font.")
        print("      Download Kh-Menghorn and place the .ttf there to enable it.")
        return ""
    b64 = base64.b64encode(FONT_PATH.read_bytes()).decode()
    print(f"Font loaded: {FONT_PATH.name} ({len(b64)//1024} KB base64)")
    return (
        "@font-face {\n"
        f"  font-family: '{FONT_FAMILY}';\n"
        f"  src: url(data:font/ttf;base64,{b64}) format('truetype');\n"
        "  font-display: swap;\n"
        "}\n"
        "*, .gradio-container, .prose, textarea, input, button, .message, .bubble {\n"
        f"  font-family: '{FONT_FAMILY}', 'Noto Sans Khmer', 'Khmer OS', sans-serif !important;\n"
        "}\n"
    )

FONT_CSS = build_font_css()


demo = gr.ChatInterface(
    fn=chat_fn,
    title="ប្រវត្តិសាស្ត្រខ្មែរ RAG — Khmer History Chatbot",
    description="សួរសំណួរជាភាសាខ្មែរអំពីប្រវត្តិសាស្ត្រកម្ពុជា។ (Ask in Khmer about Cambodian history.)",
    examples=[
        "តើអង្គរវត្តត្រូវបានសាងសង់នៅសតវត្សទីប៉ុន្មាន?",
        "នគរហ្វូណនគឺអ្វី?",
        "ខ្មែរក្រហមគឺអ្វី?",
    ],
)

demo.launch(inbrowser=True, css=FONT_CSS)   # add share=True for a public link

NOTE: font not found at d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\fonts\Kh-Menghorn.ttf -> using default Khmer font.
      Download Kh-Menghorn and place the .ttf there to enable it.
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\blocks.py", line 2698, in process_api
    result = await self.call_function(
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
  File "d:\School\AUPP\2026\Fall\NLP\Khmer_History_retrieval_argumented_generation\.venv\lib\site-packages\gradio\utils.py", line 1083, in async_wrapper
    re